In [2]:
import requests

# My API key
api_key = "fp_b1240f0a75d60c45b30f200372d5b9206b866b053de347d168dc9e00abd438da"

# API endpoint
url = "https://fireping.net/api/v1/locations"  

# Headers for authentication
headers = {
    "Authorization": f"Bearer {api_key}"
}

# Make the GET request
response = requests.get(url, headers=headers)

print(response.status_code)
print(response.json())


200
{'data': [{'id': '6c8504da-0453-4c7c-a826-b9e08a1f5b89', 'name': 'Zürich', 'latitude': 47.378261, 'longitude': 8.542042, 'radius': 10000, 'enabled': True}, {'id': '142129a4-9a4e-432f-aa92-e71b6f4d0518', 'name': 'California Test', 'latitude': 34.05, 'longitude': -118.25, 'radius': 25000, 'enabled': True}]}


In [7]:
payload = {
    "latitude": 34.05,
    "longitude": -118.25,
    "name": "California Test",
    "radius": 25000   
}

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.post(url, headers=headers, json=payload)

print(response.status_code)
data = response.json()
print(data)

#id = 142129a4-9a4e-432f-aa92-e71b6f4d0518

201
{'data': {'id': 'c3ad72b8-16ab-417a-b30d-d0139ae0f39f', 'name': 'California Test', 'latitude': 34.05, 'longitude': -118.25, 'radius': 25000, 'enabled': True}}


In [3]:
import requests

location_id = "142129a4-9a4e-432f-aa92-e71b6f4d0518"

# Append query string directly in URL
url = "https://fireping.net/api/v1/fires/user?hours=168&limit=100"

headers = {
    "Authorization": f"Bearer {api_key}"
}

response = requests.get(url, headers=headers)

print(response.status_code)
fires = response.json()
print(fires)

200
{'data': [{'latitude': 34.15114, 'longitude': -118.1948, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 2.18, 'satellite': 'N21'}, {'latitude': 34.24094, 'longitude': -118.38189, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 0.62, 'satellite': 'N21'}, {'latitude': 33.85131, 'longitude': -118.33398, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 2.19, 'satellite': 'N21'}, {'latitude': 33.85154, 'longitude': -118.33143, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 1.14, 'satellite': 'N21'}, {'latitude': 34.03581, 'longitude': -118.10569, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 0.73, 'satellite': 'N21'}, {'latitude': 34.03593, 'longitude': -118.10358, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 0.67, 'satellite': 'N21'}, {'latitude': 34.15088, 'longitude': -118.19202, 'detected_at': '2026-05-07T09:26:00Z', 'confidence': 'n', 'frp': 0.63, 'satellite': 'N21'}, {'latit

In [ ]:
import pandas as pd
import folium

df = pd.DataFrame(fires["data"])

custom_tile_url = "https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}.png"

custom_attribution = (
    '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap - yeah</a> contributors '
    '&copy; <a href="https://carto.com/attributions">CARTOoooo</a>'
)

confidence_map = {"l": "Low", "n": "Nominal", "h": "High"}
color_map = {"l": "blue", "n": "orange", "h": "red"}
radius_m = 25000

m = folium.Map(
    location=[34.05, -118.25], 
    zoom_start=10,
    tiles = custom_tile_url,
    attr= custom_attribution,
    )

# Create separate FeatureGroups for each confidence level
low_group = folium.FeatureGroup(name="Low Confidence").add_to(m)
nominal_group = folium.FeatureGroup(name="Nominal Confidence").add_to(m)
high_group = folium.FeatureGroup(name="High Confidence").add_to(m)

group_map = {"l": low_group, "n": nominal_group, "h": high_group}

# Add each fire as a circle marker
for _, fire in df.iterrows(): #loop over each row in a DataFrame

    dt = fire['detected_at'].split("T") ## Split date and time
    date = dt[0]
    time = dt[1].replace("Z", "")

    conf_text = confidence_map.get(fire["confidence"], fire["confidence"])# Convert confidence, if not find l,n,h go to default
    color = color_map.get(fire["confidence"], "gray") #if not find, b,o,r then default gray

    popup_text = (
        f"Date: {date}<br>"
        f"Time: {time}<br>"
        f"Confidence: {conf_text}<br>"
        f"FRP: {fire['frp']} MW<br>"
        f"Satellite: {fire['satellite']}"
    )
    
    folium.CircleMarker(
        location=[fire['latitude'], fire['longitude']],
        radius=6,
        popup=popup_text,
        color=color,
        fill=True,
        fill_opacity=0.7
    ).add_to(m).add_to(group_map[fire["confidence"]])

folium.Circle(
    location=[34.05, -118.25],
    radius=radius_m,       # in meters
    color="blue",
    fill=True,
    fill_opacity=0.1,
    popup="Monitoring Area: 25 km radius"
).add_to(m)

# Display map in Jupyter
m
